# Memoria Viva de Feminicidios — mapa por estado (demo)

**Capa:** mujeres muertas por presunto homicidio según el certificado de defunción (INEGI, Estadísticas de Defunciones Registradas).
Esto **no** es un conteo de feminicidios: es el universo de mujeres asesinadas, antes de cualquier investigación penal.

**Antes de empezar:** descarga los `.zip` de datos abiertos de la EDR (2023, 2024 y 2025 alcanzan) desde
https://www.inegi.org.mx/programas/edr/#datos_abiertos

Corre las celdas en orden (*Entorno de ejecución → Ejecutar todas*).

## 1. Subir los archivos del INEGI
Selecciona los `.zip` de la EDR. Si pesan mucho, usa la opción de Google Drive (celda comentada).

In [ ]:
import os, shutil
os.makedirs('datos_edr', exist_ok=True)

from google.colab import files
subidos = files.upload()
for nombre in subidos:
    shutil.move(nombre, os.path.join('datos_edr', nombre))

# --- Alternativa con Google Drive: comenta lo de arriba y descomenta esto ---
# from google.colab import drive
# drive.mount('/content/drive')
# for f in os.listdir('/content/drive/MyDrive/datos_edr'):
#     shutil.copy(os.path.join('/content/drive/MyDrive/datos_edr', f), 'datos_edr')

print(os.listdir('datos_edr'))

## 2. Script de procesamiento
Es el mismo `scripts/edr_homicidios_mujeres.py` del repositorio. Las decisiones de método están documentadas al inicio.

In [ ]:
%%writefile edr_homicidios_mujeres.py
"""
Homicidios de mujeres por municipio y año — INEGI, Estadísticas de Defunciones Registradas (EDR).

Capa "universo total de mujeres asesinadas" para el mapa de Memoria Viva de Feminicidios.

USO
  1. Descarga los datos abiertos de la EDR (un .zip por año de registro) desde
     https://www.inegi.org.mx/programas/edr/#datos_abiertos
     y ponlos todos en una carpeta, p. ej. ./datos_edr/
  2. Corre:
       python edr_homicidios_mujeres.py ./datos_edr --salida ./salida
     Opcional, para obtener un GeoJSON listo para el mapa:
       python edr_homicidios_mujeres.py ./datos_edr --salida ./salida --geojson municipios.geojson
     (el GeoJSON de municipios debe tener la propiedad CVEGEO de 5 dígitos, como el Marco Geoestadístico del INEGI)

DECISIONES METODOLÓGICAS (documentadas para que sean auditables)
  - Mujer:            sexo == 2
  - Presunto homicidio: presunto == 2  (criterio que usa el INEGI en sus reportes de homicidio).
                      Se reporta además cuántos de esos casos tienen causa CIE-10 X85–Y09 (agresiones)
                      como control de consistencia.
  - Año:              año de OCURRENCIA (anio_ocur), no de registro. Cada archivo anual del INEGI
                      contiene las defunciones REGISTRADAS ese año, incluidas algunas ocurridas antes;
                      por eso se juntan todos los archivos y se reagrupa por año de ocurrencia.
                      El año más reciente siempre estará incompleto (registros tardíos).
  - Lugar:            por defecto, municipio de ocurrencia (ent_ocurr + mun_ocurr). Se puede cambiar
                      con --lugar ocules (lugar de la lesión) o --lugar resid (residencia).
  - Unidad:           PERSONAS (cada registro es una defunción), no delitos ni carpetas.
  - Esto NO es un conteo de feminicidios: es el universo de mujeres muertas por presunto homicidio,
                      según el certificado de defunción, antes de cualquier investigación penal.
"""

import argparse
import glob
import io
import json
import os
import zipfile

import pandas as pd

COLUMNAS = [
    "sexo", "presunto", "causa_def", "anio_ocur", "anio_regis", "edad",
    "ent_ocurr", "mun_ocurr", "ent_ocules", "mun_ocules", "ent_resid", "mun_resid",
]


# ---------------------------------------------------------------- lectura
def _leer_csv(fuente):
    """Lee solo las columnas necesarias; tolera mayúsculas y codificación latin-1."""
    for enc in ("utf-8", "latin-1"):
        try:
            if hasattr(fuente, "seek"):
                fuente.seek(0)
            return pd.read_csv(
                fuente,
                usecols=lambda c: c.strip().lower() in COLUMNAS,
                dtype=str,
                encoding=enc,
                low_memory=False,
            )
        except UnicodeDecodeError:
            continue
    raise ValueError("No se pudo leer el archivo con utf-8 ni latin-1")


def leer_archivo(ruta):
    """Acepta .csv o .zip de datos abiertos del INEGI (busca dentro el CSV de defunciones)."""
    if ruta.lower().endswith(".zip"):
        with zipfile.ZipFile(ruta) as z:
            csvs = [n for n in z.namelist()
                    if n.lower().endswith(".csv") and "defun" in os.path.basename(n).lower()]
            if not csvs:  # algunos zips nombran distinto el archivo
                csvs = [n for n in z.namelist()
                        if n.lower().endswith(".csv") and "conjunto_de_datos" in n.lower()]
            if not csvs:
                raise ValueError(f"No encontré el CSV de defunciones dentro de {ruta}")
            # el más grande es la base; los demás suelen ser catálogos
            nombre = max(csvs, key=lambda n: z.getinfo(n).file_size)
            df = _leer_csv(io.BytesIO(z.read(nombre)))
    else:
        df = _leer_csv(ruta)
    df.columns = [c.strip().lower() for c in df.columns]
    df["archivo"] = os.path.basename(ruta)
    return df


# ---------------------------------------------------------------- limpieza
def a_num(s):
    return pd.to_numeric(s, errors="coerce")


def es_agresion_cie10(causa):
    """True si la causa CIE-10 está entre X85 y Y09 (agresiones)."""
    c = causa.fillna("").str.upper().str.strip().str[:3]
    letra, num = c.str[0], pd.to_numeric(c.str[1:3], errors="coerce")
    return ((letra == "X") & (num >= 85)) | ((letra == "Y") & (num <= 9))


def es_arma_fuego(causa):
    """X93, X94, X95: agresión con disparo de arma de fuego."""
    return causa.fillna("").str.upper().str[:3].isin(["X93", "X94", "X95"])


def edad_en_anios(edad):
    """En la EDR, la edad en años se codifica 4001–4120 (4000 + años). Otros códigos = menores de 1 año o no especificado."""
    e = a_num(edad)
    anios = pd.Series(pd.NA, index=e.index, dtype="Int64")
    anios[(e >= 4000) & (e <= 4120)] = (e - 4000)[(e >= 4000) & (e <= 4120)].astype("Int64")
    anios[(e > 0) & (e < 4000)] = 0  # horas, días o meses -> menor de un año
    return anios


def cvegeo(ent, mun):
    """Clave geoestadística de 5 dígitos. Entidad >32 o municipio 999 se marcan como no especificado."""
    e, m = a_num(ent), a_num(mun)
    valido = e.between(1, 32) & m.between(1, 998)
    clave = e.astype("Int64").astype(str).str.zfill(2) + m.astype("Int64").astype(str).str.zfill(3)
    clave[~valido] = "NO_ESPECIFICADO"
    return clave, e.astype("Int64").astype(str).str.zfill(2).where(e.between(1, 32), "NO_ESPECIFICADO")


# ---------------------------------------------------------------- proceso
def procesar(carpeta, lugar="ocurr"):
    rutas = sorted(glob.glob(os.path.join(carpeta, "*.zip")) + glob.glob(os.path.join(carpeta, "*.csv")))
    if not rutas:
        raise SystemExit(f"No hay archivos .zip o .csv en {carpeta}")

    partes = []
    for r in rutas:
        df = leer_archivo(r)
        faltan = {"sexo", "presunto", "anio_ocur", f"ent_{lugar}", f"mun_{lugar}"} - set(df.columns)
        if faltan:
            print(f"  ! {os.path.basename(r)}: faltan columnas {sorted(faltan)}; se omite")
            continue
        n_total = len(df)
        df = df[(a_num(df["sexo"]) == 2) & (a_num(df["presunto"]) == 2)].copy()
        print(f"  {os.path.basename(r)}: {n_total:,} defunciones -> {len(df):,} mujeres por presunto homicidio")
        partes.append(df)

    datos = pd.concat(partes, ignore_index=True)
    datos["anio"] = a_num(datos["anio_ocur"]).astype("Int64")
    datos["cvegeo"], datos["cve_ent"] = cvegeo(datos[f"ent_{lugar}"], datos[f"mun_{lugar}"])
    datos["agresion_cie10"] = es_agresion_cie10(datos.get("causa_def", pd.Series(index=datos.index, dtype=str)))
    datos["arma_fuego"] = es_arma_fuego(datos.get("causa_def", pd.Series(index=datos.index, dtype=str)))
    edad = edad_en_anios(datos.get("edad", pd.Series(index=datos.index, dtype=str)))
    datos["menor_18"] = (edad < 18).fillna(False)
    datos["edad_no_esp"] = edad.isna()
    # año de ocurrencia desconocido (9999) se excluye de la serie, pero se reporta
    sin_anio = (~datos["anio"].between(1990, 2100)).sum()
    datos = datos[datos["anio"].between(1990, 2100)]
    return datos, sin_anio


def agregar(datos, nivel):
    llave = "cvegeo" if nivel == "municipio" else "cve_ent"
    return (datos.groupby([llave, "anio"])
            .agg(homicidios_mujeres=("sexo", "size"),
                 con_causa_x85_y09=("agresion_cie10", "sum"),
                 arma_fuego=("arma_fuego", "sum"),
                 menores_18=("menor_18", "sum"),
                 edad_no_especificada=("edad_no_esp", "sum"))
            .reset_index()
            .sort_values([llave, "anio"]))


def a_geojson(mun, ruta_geo, ruta_salida):
    """Une la tabla municipal (formato ancho, una columna por año) al GeoJSON de municipios por CVEGEO."""
    ancho = mun.pivot(index="cvegeo", columns="anio", values="homicidios_mujeres").fillna(0).astype(int)
    ancho.columns = [f"h_{a}" for a in ancho.columns]
    tabla = ancho.to_dict(orient="index")
    with open(ruta_geo, encoding="utf-8") as f:
        geo = json.load(f)
    sin_match = 0
    for feat in geo["features"]:
        props = feat.setdefault("properties", {})
        clave = str(props.get("CVEGEO") or props.get("cvegeo") or "").zfill(5)
        valores = tabla.get(clave)
        if valores is None:
            sin_match += 1
            valores = {c: 0 for c in ancho.columns}
        props.update(valores)
    with open(ruta_salida, "w", encoding="utf-8") as f:
        json.dump(geo, f, ensure_ascii=False)
    claves_geo = {str(ft["properties"].get("CVEGEO") or ft["properties"].get("cvegeo") or "").zfill(5)
                  for ft in geo["features"]}
    huerfanas = set(tabla) - claves_geo
    print(f"  GeoJSON: {len(geo['features'])} municipios, {sin_match} sin casos; "
          f"{len(huerfanas)} claves de la tabla no encontradas en el GeoJSON")


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("carpeta", help="Carpeta con los .zip o .csv de datos abiertos de la EDR")
    p.add_argument("--salida", default="salida")
    p.add_argument("--lugar", choices=["ocurr", "ocules", "resid"], default="ocurr",
                   help="ocurr = donde ocurrió la defunción (default); ocules = donde ocurrió la lesión; resid = residencia")
    p.add_argument("--geojson", help="GeoJSON de municipios con propiedad CVEGEO (opcional)")
    a = p.parse_args()

    os.makedirs(a.salida, exist_ok=True)
    print("Leyendo archivos…")
    datos, sin_anio = procesar(a.carpeta, a.lugar)

    mun = agregar(datos, "municipio")
    est = agregar(datos, "estado")
    mun.to_csv(os.path.join(a.salida, "homicidios_mujeres_municipio_anio.csv"), index=False)
    est.to_csv(os.path.join(a.salida, "homicidios_mujeres_estado_anio.csv"), index=False)

    nacional = datos.groupby("anio").size()
    print("\nMujeres por presunto homicidio, por año de ocurrencia (nacional):")
    print(nacional.to_string())
    no_esp = (datos["cvegeo"] == "NO_ESPECIFICADO").sum()
    inconsist = (~datos["agresion_cie10"]).sum()
    print(f"\nControles de calidad:")
    print(f"  municipio no especificado: {no_esp:,} ({no_esp / len(datos):.1%})")
    print(f"  año de ocurrencia no especificado (excluidos): {sin_anio:,}")
    print(f"  presunto homicidio con causa fuera de X85–Y09: {inconsist:,}")
    print(f"  año más reciente ({nacional.index.max()}) está incompleto por registros tardíos")

    if a.geojson:
        a_geojson(mun[mun["cvegeo"] != "NO_ESPECIFICADO"], a.geojson,
                  os.path.join(a.salida, "homicidios_mujeres_municipios.geojson"))
    print(f"\nListo. Archivos en {a.salida}/")


if __name__ == "__main__":
    main()


## 3. Procesar

In [ ]:
import importlib, edr_homicidios_mujeres as edr
importlib.reload(edr)

datos, sin_anio = edr.procesar('datos_edr', lugar='ocurr')
est = edr.agregar(datos, 'estado')
est = est[est['cve_ent'] != 'NO_ESPECIFICADO']

nacional = datos.groupby('anio').size()
print('Mujeres por presunto homicidio, por año de ocurrencia:')
print(nacional.to_string())
print(f"\nAño de ocurrencia no especificado (excluidos): {sin_anio}")
print('\nCONTROL: compara contra lo publicado por el INEGI (p. ej. 3,928 mujeres en 2022).')
print('Si la cifra es muy distinta, revisa los códigos de sexo/presunto en el catálogo del zip.')
print('Los años viejos solo traen registros tardíos y el más reciente está incompleto: usa los años completos.')

## 4. Años a mostrar
Solo años con datos completos. Un año de ocurrencia queda casi completo cuando ya tienes cargado el archivo del año siguiente.

In [ ]:
ANIOS = sorted(int(a) for a in est['anio'].unique() if a in set(nacional.index) and nacional[a] > 500)
print('Años en el mapa:', ANIOS)

## 5. (Opcional) Agregar feminicidios del SESNSP
Si tienes el CSV de **víctimas** del fuero común a nivel estatal del SESNSP, súbelo aquí. Si no, sáltate esta celda: el mapa funciona sin ella.
La celda busca las columnas por nombre; si el formato no coincide, avisa y sigue sin esta capa.

In [ ]:
import pandas as pd
sesnsp = None
USAR_SESNSP = False   # cambia a True para subir el archivo

if USAR_SESNSP:
    arch = files.upload()
    ruta = list(arch)[0]
    try:
        df = pd.read_csv(ruta, encoding='latin-1')
        cols = {c.lower().strip(): c for c in df.columns}
        c_anio = next(cols[k] for k in cols if k in ('año', 'ano', 'anio'))
        c_ent = next(cols[k] for k in cols if 'clave_ent' in k or k == 'cve_ent')
        c_tipo = next(cols[k] for k in cols if 'tipo de delito' in k)
        meses = [cols[k] for k in cols if k in ('enero','febrero','marzo','abril','mayo','junio','julio',
                                                'agosto','septiembre','octubre','noviembre','diciembre')]
        f = df[df[c_tipo].astype(str).str.lower().str.contains('feminicidio')].copy()
        f['total'] = f[meses].apply(pd.to_numeric, errors='coerce').sum(axis=1)
        sesnsp = (f.groupby([c_ent, c_anio])['total'].sum().reset_index()
                    .rename(columns={c_ent: 'cve_ent', c_anio: 'anio', 'total': 'feminicidios_sesnsp'}))
        sesnsp['cve_ent'] = sesnsp['cve_ent'].astype(int).astype(str).str.zfill(2)
        sesnsp['anio'] = sesnsp['anio'].astype(int)
        print(sesnsp.groupby('anio')['feminicidios_sesnsp'].sum())
    except Exception as e:
        print('No pude leer el formato del SESNSP, sigo sin esa capa:', e)
        sesnsp = None

## 6. Mapa

In [ ]:
import json, urllib.request, folium, branca.colormap as cm

URL_ESTADOS = 'https://raw.githubusercontent.com/strotgen/mexico-leaflet/master/states.geojson'
geo = json.load(urllib.request.urlopen(URL_ESTADOS))

tabla = est.copy()
if sesnsp is not None:
    tabla = tabla.merge(sesnsp, on=['cve_ent', 'anio'], how='left')

NOMBRES = {9: 'Ciudad de México', 5: 'Coahuila', 16: 'Michoacán', 30: 'Veracruz'}
for ft in geo['features']:
    p = ft['properties']
    p['nombre'] = NOMBRES.get(p['state_code'], p['state_name'])
    p['cve'] = str(p['state_code']).zfill(2)

vmax = tabla[tabla['anio'].isin(ANIOS)]['homicidios_mujeres'].max()
escala = cm.LinearColormap(['#fbe9e7', '#e57373', '#b71c1c', '#4a0000'], vmin=0, vmax=vmax,
                           caption='Mujeres muertas por presunto homicidio (INEGI, EDR)')

m = folium.Map(location=[23.6, -102.5], zoom_start=5, tiles=None, control_scale=True)
folium.TileLayer('OpenStreetMap', control=False, opacity=0.5).add_to(m)

for i, anio in enumerate(ANIOS):
    t = tabla[tabla['anio'] == anio].set_index('cve_ent')
    capa = json.loads(json.dumps(geo))
    for ft in capa['features']:
        p = ft['properties']
        fila = t.loc[p['cve']] if p['cve'] in t.index else None
        val = lambda col: int(fila[col]) if fila is not None and col in fila and pd.notna(fila[col]) else 0
        p['homicidios'] = val('homicidios_mujeres')
        p['arma_fuego'] = val('arma_fuego')
        p['menores_18'] = val('menores_18')
        if sesnsp is not None:
            p['feminicidios'] = val('feminicidios_sesnsp')
            p['brecha'] = p['homicidios'] - p['feminicidios']
    campos = ['nombre', 'homicidios', 'arma_fuego', 'menores_18']
    alias = ['Estado', 'Mujeres asesinadas (INEGI)', 'Con arma de fuego', 'Menores de 18']
    if sesnsp is not None:
        campos += ['feminicidios', 'brecha']
        alias += ['Feminicidios (SESNSP)', 'No registradas como feminicidio']
    folium.GeoJson(
        capa, name=str(anio), overlay=False, show=(i == len(ANIOS) - 1),
        style_function=lambda f: {'fillColor': escala(f['properties']['homicidios']),
                                  'color': '#555', 'weight': 0.6, 'fillOpacity': 0.85},
        highlight_function=lambda f: {'weight': 2, 'color': '#000'},
        tooltip=folium.GeoJsonTooltip(fields=campos, aliases=alias, localize=True),
    ).add_to(m)

escala.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

titulo = '''<div style="position:fixed;top:12px;left:60px;z-index:9999;background:white;padding:8px 14px;
border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font-family:sans-serif;max-width:420px">
<b>Memoria Viva de Feminicidios</b><br>
<span style="font-size:12px">Mujeres muertas por presunto homicidio, por estado de ocurrencia.
Fuente: INEGI, Estadísticas de Defunciones Registradas. No es un conteo de feminicidios.
Elige el año en el control de la derecha.</span></div>'''
m.get_root().html.add_child(folium.Element(titulo))
m

## 7. Exportar
Descarga `index.html` y súbelo a la carpeta `web/` del repositorio para publicarlo en GitHub Pages.

In [ ]:
m.save('index.html')
files.download('index.html')